#### Combine All Cleaned Datasets

Joins all tables in `cleaned_data/` on `(participant_id, timepoint)`. Only participants present in the HAI table are kept. All joins use `how='outer'` — missing values become NaN. Final table is filtered to timepoints `[0, 7, 28, 365]`.

In [1]:
import os
import pandas as pd
from IPython.display import display

CLEANED_DATA_DIR = 'cleaned_data'
OUTPUT_DIR = 'merged_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 1. Participants — no timepoint, demographics broadcast via `participant_id`

In [ ]:
participants = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'participants_cleaned.csv'))
print(f"Participants shape: {participants.shape}")

#### 2. HAI — pivot `virus_strain` to columns

In [ ]:
hai = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'hai_cleaned.csv'))
print(f"HAI shape: {hai.shape}")

# Define the participant filter — all subsequent tables are restricted to this set
hai_participants = set(hai['participant_id'].unique())
print(f"Unique participants in HAI: {len(hai_participants)}")

# Filter participants to HAI cohort
participants = participants[participants['participant_id'].isin(hai_participants)].reset_index(drop=True)
print(f"Participants after filter:  {participants.shape}")

hai_pivot = hai.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='virus_strain',
    values='value',
    aggfunc='mean'
).reset_index()
hai_pivot.columns.name = None
hai_pivot.rename(columns={
    c: f'hai_{c}' for c in hai_pivot.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

# Drop the 'hai_-' column if present — virus_strain='-' is a data artifact
if 'hai_-' in hai_pivot.columns:
    hai_pivot.drop(columns=['hai_-'], inplace=True)

print(f"\nHAI pivoted shape: {hai_pivot.shape}")
display(hai_pivot.head(3))

#### 3. Flow Cytometry — pivot cell population `name` to columns

In [ ]:
flow = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'flow_cleaned.csv'), low_memory=False)
flow = flow[flow['participant_id'].isin(hai_participants)]
print(f"Flow shape (filtered): {flow.shape}")

flow_pivot = flow.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='name',
    values='value',
    aggfunc='mean'
).reset_index()
flow_pivot.columns.name = None

def clean_col_name(name):
    """Sanitize cell population names for use as column names."""
    return (name.replace(' ', '_')
                .replace('+', 'pos')
                .replace('-', 'neg')
                .replace('/', '_')
                .replace('%', 'pct')
                .replace('(', '')
                .replace(')', ''))

flow_pivot.rename(columns={
    c: f'flow_{clean_col_name(c)}' for c in flow_pivot.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

print(f"Flow pivoted shape:    {flow_pivot.shape}")
display(flow_pivot.iloc[:3, :6])

#### 4. Microarray — pivot `gene_symbol` to columns (mean across platforms)

In [ ]:
array = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'array_cleaned.csv'))
array = array[array['participant_id'].isin(hai_participants)]
print(f"Array shape (filtered): {array.shape}")

array_pivot = array.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='gene_symbol',
    values='value',
    aggfunc='mean'
).reset_index()
array_pivot.columns.name = None
array_pivot.rename(columns={
    c: f'array_{c}' for c in array_pivot.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

print(f"Array pivoted shape:    {array_pivot.shape}")
display(array_pivot.iloc[:3, :6])

#### 5. BCR — aggregate numeric columns (mean) per `(participant_id, timepoint)` + sequence count

In [ ]:
bcr = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'bcr_cleaned.csv'), low_memory=False)
bcr = bcr[bcr['participant_id'].isin(hai_participants)]
print(f"BCR shape (filtered): {bcr.shape}")

exclude = {'participant_id', 'timepoint', 'sex', 'age', 'biological_sex',
           'subject_id', 'sample_id', 'clone_id'}
bcr_numeric_cols = [
    c for c in bcr.select_dtypes(include='number').columns if c not in exclude
]

bcr_agg = (bcr
           .groupby(['participant_id', 'timepoint'])[bcr_numeric_cols]
           .mean()
           .reset_index())

bcr_counts = (bcr
              .groupby(['participant_id', 'timepoint'])
              .size()
              .reset_index(name='sequence_count'))

bcr_agg = bcr_agg.merge(bcr_counts, on=['participant_id', 'timepoint'])
bcr_agg.rename(columns={
    c: f'bcr_{c}' for c in bcr_agg.columns if c not in ['participant_id', 'timepoint']
}, inplace=True)

print(f"BCR aggregated shape: {bcr_agg.shape}")
display(bcr_agg.head(3))

#### 6. Transcriptomics — concat 4 datasets (outer join aligns gene columns, fills missing with NaN)

In [ ]:
tx_files = {
    '2019_UGA': 'transcriptomics_2019_UGA_cleaned.csv',
    '2020_UGA': 'transcriptomics_2020_UGA_cleaned.csv',
    'SDY2867':  'transcriptomics_SDY2867_cleaned.csv',
    'SDY2941':  'transcriptomics_SDY2941_cleaned.csv',
}

tx_dfs = []
for label, fname in tx_files.items():
    df = pd.read_csv(os.path.join(CLEANED_DATA_DIR, fname))
    df = df[df['participant_id'].isin(hai_participants)]
    gene_cols = [c for c in df.columns if c not in ('participant_id', 'timepoint')]
    print(f"{label}: {df.shape[0]} rows × {len(gene_cols)} gene columns")
    tx_dfs.append(df)

transcriptomics = pd.concat(tx_dfs, axis=0, join='outer').reset_index(drop=True)
gene_cols_all = [c for c in transcriptomics.columns if c not in ('participant_id', 'timepoint')]
print(f"\nCombined transcriptomics: {transcriptomics.shape[0]} rows × {len(gene_cols_all)} gene columns")

#### 7. Merge — union all `(participant_id, timepoint)` pairs, then outer-join each table

In [8]:
timepoint_tables = {
    'HAI':             hai_pivot,
    'Flow':            flow_pivot,
    'Array':           array_pivot,
    'BCR':             bcr_agg,
    'Transcriptomics': transcriptomics,
}

# Step 1: union of all (participant_id, timepoint) pairs
all_pairs = pd.concat(
    [t[['participant_id', 'timepoint']] for t in timepoint_tables.values()],
    ignore_index=True
).drop_duplicates().reset_index(drop=True)
print(f"Total unique (participant_id, timepoint) pairs: {len(all_pairs)}")

# Step 2: broadcast participant demographics to every timepoint row
merged = all_pairs.merge(participants, on='participant_id', how='left')
print(f"After joining participants:   {merged.shape}")

# Step 3: outer-join each feature table
for name, table in timepoint_tables.items():
    merged = merged.merge(table, on=['participant_id', 'timepoint'], how='outer')
    print(f"After joining {name:<15}: {merged.shape}")

print(f"\nFinal combined dataset: {merged.shape[0]:,} rows × {merged.shape[1]:,} columns")

Total unique (participant_id, timepoint) pairs: 11504
After joining participants:   (11504, 6)
After joining HAI            : (11504, 74)
After joining Flow           : (11504, 102)
After joining Array          : (11504, 39767)
After joining BCR            : (11504, 39799)
After joining Transcriptomics: (11504, 119023)

Final combined dataset: 11,504 rows × 119,023 columns


#### 8. Column Summary

In [9]:
col_groups = {
    'Keys':            [c for c in ['participant_id', 'timepoint'] if c in merged.columns],
    'Demographics':    [c for c in participants.columns if c != 'participant_id'],
    'HAI':             [c for c in merged.columns if c.startswith('hai_')],
    'Flow':            [c for c in merged.columns if c.startswith('flow_')],
    'Array':           [c for c in merged.columns if c.startswith('array_')],
    'BCR':             [c for c in merged.columns if c.startswith('bcr_')],
    'Transcriptomics': [c for c in merged.columns if c.startswith('ENSG')],
}

print("Column counts by group:")
for group, cols in col_groups.items():
    print(f"  {group:<18}: {len(cols):>6}")
print(f"  {'TOTAL':<18}: {sum(len(v) for v in col_groups.values()):>6}")

print("\nSample rows (key + demographic columns):")
preview_cols = col_groups['Keys'] + col_groups['Demographics']
display(merged[preview_cols].head(5))

Column counts by group:
  Keys              :      2
  Demographics      :      4
  HAI               :     68
  Flow              :     28
  Array             :  39665
  BCR               :     32
  Transcriptomics   :  79224
  TOTAL             : 119023

Sample rows (key + demographic columns):


,participant_id,timepoint,biological_sex,race,geolocation,age
0,2016_UGA.ID_001,0.0,Female,Unknown,Georgia,29.0
1,2016_UGA.ID_001,28.0,Female,Unknown,Georgia,29.0
2,2016_UGA.ID_002,0.0,Female,Unknown,Georgia,29.0
3,2016_UGA.ID_002,28.0,Female,Unknown,Georgia,29.0
4,2016_UGA.ID_003,0.0,Female,Unknown,Georgia,28.0


#### 10. Save

#### 9. Filter Timepoints

In [ ]:
KEEP_TIMEPOINTS = [0, 7, 28, 365]

print(f"Timepoints before filter: {sorted(merged['timepoint'].dropna().unique())}")
merged = merged[merged['timepoint'].isin(KEEP_TIMEPOINTS)].reset_index(drop=True)
print(f"Timepoints after filter:  {sorted(merged['timepoint'].unique())}")
print(f"Shape after filter: {merged.shape[0]:,} rows × {merged.shape[1]:,} columns")

In [10]:
# output_path = os.path.join(OUTPUT_DIR, 'combined_dataset.csv')
# merged.to_csv(output_path, index=False)
# print(f"Saved: {output_path}")
# print(f"Shape: {merged.shape[0]:,} rows × {merged.shape[1]:,} columns")

In [11]:
merged.head()

,participant_id,timepoint,biological_sex,race,geolocation,age,hai_Anc B/Lee/1940,hai_Anc B/Maryland/1959,hai_Anc B/Singapore/1964,hai_H1N1 A/Beijing/262/1995,...,ENSG00000310527,ENSG00000310529,ENSG00000310530,ENSG00000310532,ENSG00000310533,ENSG00000310534,ENSG00000310535,ENSG00000310536,ENSG00000310537,ENSG00000310539
0,2016_UGA.ID_001,0.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016_UGA.ID_001,28.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016_UGA.ID_002,0.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016_UGA.ID_002,28.0,Female,Unknown,Georgia,29.0,NaN,NaN,NaN,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016_UGA.ID_003,0.0,Female,Unknown,Georgia,28.0,NaN,NaN,NaN,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
